# Construindo o pipeline do zero: do CSV ao DuckDB

Este notebook é uma aula guiada: você vai construir, célula por célula, as mesmas peças que já existem em `scripts/extract_csv.py`, `scripts/extract_weather.py` e `scripts/transform.sql`, só que aqui dá pra parar em cada passo e entender o "porquê" antes de ver a versão final do script.

**Importante:** este notebook grava num banco **separado** (`warehouse/aula_pipeline.duckdb`), não no `workshop.duckdb` que alimenta o Metabase: pode rodar, quebrar e rodar de novo sem medo de estragar a dashboard da aula.

Roteiro:
1. Ler um CSV
2. O problema: duas fontes, dois esquemas
3. Renomear colunas
4. Transformar isso numa função reutilizável
5. Escrever o resultado no DuckDB
6. Fazer uma requisição numa API externa
7. Repetir a requisição para todas as cidades
8. Juntar tudo num mini pipeline, o mesmo desenho do `run_pipeline.py`
9. Staging em SQL puro: reconciliar e limpar as vendas
10. Staging em SQL puro: clima
11. Mart: responder a pergunta de negócio
12. Stretch: os outros marts
13. Resumo final
14. Agora é sua vez: perguntas de negócio

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import requests

RAW_CSV = Path("../data/raw/vendas_lojas.csv")
INCOMING_CSV = Path("../data/incoming/vendas_lojas_recentes.csv")

# Banco de sandbox só desta aula, separado do warehouse/workshop.duckdb
# que o Metabase usa, pra podermos experimentar sem medo de estragar nada
DB_PATH = Path("../warehouse/aula_pipeline.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Lendo um CSV

`pandas.read_csv` já lê pelo **nome** da coluna (o header), não pela posição. Isso importa porque, como vamos ver já na próxima seção, as duas fontes de venda da GeloDados têm colunas em nomes e ordens diferentes: ler por posição quebraria silenciosamente.

In [ ]:
# dtype=str: deixamos tudo como texto por enquanto, igual ao CSV cru.
# A tipagem (converter pra número, data etc.) é trabalho da staging, mais
# pra frente, não da leitura em si.
historico = pd.read_csv(RAW_CSV, dtype=str)
historico.head()

In [ ]:
print(f"{len(historico)} linhas")
print(f"colunas: {list(historico.columns)}")


## 2. O problema: duas fontes, dois esquemas

O time comercial trocou de sistema de ponto de venda na metade da janela de dados. O arquivo `vendas_lojas_recentes.csv` tem **a mesma informação**, só que com colunas renomeadas e em outra ordem.


In [ ]:
recentes = pd.read_csv(INCOMING_CSV, dtype=str)
recentes.head()

In [ ]:
# As colunas não batem: juntar os dois DataFrames direto (pd.concat) ia
# criar um Frankenstein de colunas, a maioria cheia de NaN
set(historico.columns) ^ set(recentes.columns)

## 3. Renomeando colunas

A solução: um dicionário explícito de `nome_na_fonte -> nome_canônico`, igual ao que `scripts/transform.sql` faz em SQL. Aqui fazemos a mesma reconciliação, só que em pandas.


In [ ]:
mapeamento_recentes = {
    "dt_venda": "data_venda",
    "id_loja": "loja_id",
    "uf_cidade": "cidade",
    "id_cliente": "cliente_id",
    "nm_cliente": "nome_cliente",
    "idade": "idade_cliente",
    "genero": "genero_cliente",
    "categoria_produto": "produto",
    "sabor_item": "sabor",
    "qtd_vendida": "unidades_vendidas",
    "vl_unitario": "preco_unitario",
    "vl_total": "receita_brl",
    "tipo_pagamento": "forma_pagamento",
}

recentes_renomeado = recentes.rename(columns=mapeamento_recentes)
recentes_renomeado.head()


In [ ]:
assert set(historico.columns) == set(recentes_renomeado.columns)

vendas = pd.concat([historico, recentes_renomeado], ignore_index=True)
print(f"{len(vendas)} linhas no total, agora com um único esquema de colunas")


## 4. Transformando isso numa função

Reparou que "ler o CSV" e "renomear se precisar" é uma receita que vamos repetir para qualquer fonte nova que aparecer? Isso é sinal de que vale a pena virar uma função.


In [ ]:
def carregar_vendas(caminho_csv, mapeamento=None):
    df = pd.read_csv(caminho_csv, dtype=str)
    if mapeamento:
        df = df.rename(columns=mapeamento)
    return df


historico_v2 = carregar_vendas(RAW_CSV)
recentes_v2 = carregar_vendas(INCOMING_CSV, mapeamento=mapeamento_recentes)

vendas = pd.concat([historico_v2, recentes_v2], ignore_index=True)
vendas.sample(5)

### Pausa pra praticar

Quantas linhas de `historico_v2` têm `idade_cliente` vazia? (Lembre: estamos lendo com `dtype=str`, então "vazia" aqui pode ser string vazia `""` ou `NaN`, dependendo de como o pandas leu a célula do CSV.)

In [ ]:
# seu código aqui

## 5. Escrevendo no DuckDB

DuckDB não tem um "insira este DataFrame direto numa tabela nova": o jeito mais simples é **registrar** o DataFrame como uma view temporária (`con.register`) e usar `CREATE TABLE ... AS SELECT * FROM` para materializar.

Também usamos **schemas** (`raw`, `staging`, `mart`) pra deixar claro em que estágio de limpeza cada tabela está. Aqui só vamos até `raw`.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

con.register("_tmp_vendas", vendas)
con.execute("CREATE OR REPLACE TABLE raw.vendas AS SELECT * FROM _tmp_vendas")
con.unregister("_tmp_vendas")

con.execute("SELECT COUNT(*) AS total FROM raw.vendas").df()


Essa sequência (`register` → `CREATE OR REPLACE TABLE` → `unregister`) também vale a pena virar função: vamos usá-la de novo já na próxima seção.

In [ ]:
def salvar_no_duckdb(con, df, tabela):
    con.register("_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {tabela} AS SELECT * FROM _tmp")
    con.unregister("_tmp")
    print(f"{len(df)} linhas -> {tabela}")


# repare: salvamos "recentes" (schema original), não "recentes_v2"
# (já renomeado). O raw deve espelhar a fonte fielmente; quem reconcilia
# o nome das colunas é a camada de staging, em SQL, mais abaixo.
salvar_no_duckdb(con, historico_v2, "raw.vendas_historico")
salvar_no_duckdb(con, recentes, "raw.vendas_recentes")

## 6. Fazendo uma requisição numa API

O clima não existe nos sistemas internos da GeloDados: precisa vir de uma **API pública**, a [Open-Meteo](https://open-meteo.com/). `requests.get` com os parâmetros certos (coordenadas + janela de datas) devolve um JSON com a série diária.

In [ ]:
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": -23.5505,
        "longitude": -46.6333,
        "start_date": "2026-06-01",
        "end_date": "2026-06-10",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "America/Sao_Paulo",
    },
    timeout=10,
)
resp.raise_for_status()
dados = resp.json()
dados["daily"]


O JSON vem "achatado" em listas paralelas (uma lista de datas, uma de temperaturas máximas, etc.). É só remontar isso num DataFrame.

In [ ]:
clima_sp = pd.DataFrame({
    "cidade": "Sao Paulo",
    "data": dados["daily"]["time"],
    "temp_max_c": dados["daily"]["temperature_2m_max"],
    "temp_min_c": dados["daily"]["temperature_2m_min"],
    "precipitacao_mm": dados["daily"]["precipitation_sum"],
})
clima_sp


In [ ]:
# mesma função de sempre, não importa se o dado veio de CSV ou de API
salvar_no_duckdb(con, clima_sp, "raw.clima_sp")

## 7. Repetindo para todas as cidades

A GeloDados tem 5 lojas em 5 cidades, e a janela de vendas vai de 2026-06-01 a 2026-09-01. O teste acima foi só São Paulo, em 10 dias. Pra responder a pergunta de negócio de verdade (o clima influencia as vendas?), precisamos do clima de TODAS as cidades, na janela TODA.

Mesma lógica de sempre: vira função, chama em loop.

In [ ]:
COORDENADAS = {
    "Sao Paulo": (-23.5505, -46.6333),
    "Rio de Janeiro": (-22.9068, -43.1729),
    "Curitiba": (-25.4284, -49.2733),
    "Fortaleza": (-3.7172, -38.5433),
    "Porto Alegre": (-30.0346, -51.2177),
}


def buscar_clima(cidade, lat, lon):
    resp = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": lat,
            "longitude": lon,
            "start_date": "2026-06-01",
            "end_date": "2026-09-01",
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
            "timezone": "America/Sao_Paulo",
        },
        timeout=10,
    )
    resp.raise_for_status()
    daily = resp.json()["daily"]
    return pd.DataFrame({
        "cidade": cidade,
        "data": daily["time"],
        "temp_max_c": daily["temperature_2m_max"],
        "temp_min_c": daily["temperature_2m_min"],
        "precipitacao_mm": daily["precipitation_sum"],
    })


clima = pd.concat(
    [buscar_clima(cidade, lat, lon) for cidade, (lat, lon) in COORDENADAS.items()],
    ignore_index=True,
)
salvar_no_duckdb(con, clima, "raw.clima")

O script de verdade, `scripts/extract_weather.py`, também geocodifica cada cidade (converte o nome em lat/long via outra API) e tem fallback pra um cache local se a internet cair no meio da aula. Aqui simplificamos usando coordenadas fixas direto.

## 8. Juntando tudo: o mesmo desenho do `run_pipeline.py`

Com as peças prontas (`carregar_vendas`, `salvar_no_duckdb`), "o pipeline" é só chamar essas funções na ordem certa. Dá uma olhada em `scripts/run_pipeline.py` depois: é exatamente essa ideia, só chamando os scripts de verdade em vez de funções dentro de um notebook.

In [ ]:
def pipeline_de_vendas():
    historico = carregar_vendas(RAW_CSV)
    recentes = carregar_vendas(INCOMING_CSV)
    salvar_no_duckdb(con, historico, "raw.vendas_historico")
    salvar_no_duckdb(con, recentes, "raw.vendas_recentes")


pipeline_de_vendas()

In [ ]:
con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = 'raw'
    ORDER BY table_name
""").df()


## 9. Staging em SQL puro: reconciliando as vendas

Agora que `raw.vendas_historico` e `raw.vendas_recentes` estão no banco, dá pra fazer a MESMA reconciliação que fizemos em pandas na seção 3, só que em SQL, direto no banco. É assim que `scripts/transform.sql` faz de verdade.

Repare que `raw.vendas_recentes` guardamos com o schema ORIGINAL (`dt_venda`, `id_loja`...), sem aplicar o `mapeamento_recentes`. Foi proposital: o raw deve espelhar a fonte fielmente; quem reconcilia o nome das colunas é a staging.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE stg_vendas AS
    WITH historico_renomeado AS (
        SELECT
            data_venda, loja_id, cidade, cliente_id, nome_cliente,
            idade_cliente, genero_cliente, produto, sabor,
            unidades_vendidas, preco_unitario, receita_brl, forma_pagamento
        FROM raw.vendas_historico
    ),
    recentes_renomeado AS (
        SELECT
            dt_venda          AS data_venda,
            id_loja           AS loja_id,
            uf_cidade         AS cidade,
            id_cliente        AS cliente_id,
            nm_cliente        AS nome_cliente,
            idade             AS idade_cliente,
            genero            AS genero_cliente,
            categoria_produto AS produto,
            sabor_item        AS sabor,
            qtd_vendida       AS unidades_vendidas,
            vl_unitario       AS preco_unitario,
            vl_total          AS receita_brl,
            tipo_pagamento    AS forma_pagamento
        FROM raw.vendas_recentes
    )
    SELECT DISTINCT * FROM historico_renomeado
    UNION ALL
    SELECT DISTINCT * FROM recentes_renomeado
""")

con.execute("SELECT * FROM stg_vendas LIMIT 5").df()

`WITH` cria uma CTE (common table expression): a versão SQL de uma função, dá nome a um `SELECT` pra reusar. `SELECT DISTINCT` remove duplicatas exatas de cada fonte antes de juntar; `UNION ALL` empilha as duas.

Ainda sobra sujeira: a `data_venda` vem em dois formatos (`2026-06-01` e `01/06/2026`), tem `idade_cliente`/`genero_cliente` faltando, e tudo ainda está como texto. Dá pra limpar isso fazendo um `SELECT` sobre a própria `stg_vendas`, sem refazer o `UNION ALL`.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE stg_vendas AS
    SELECT
        CASE
            WHEN data_venda LIKE '____-__-__' THEN CAST(data_venda AS DATE)
            WHEN data_venda LIKE '__/__/____' THEN CAST(STRPTIME(data_venda, '%d/%m/%Y') AS DATE)
        END AS data_venda,
        loja_id,
        cidade,
        cliente_id,
        NULLIF(nome_cliente, '') AS nome_cliente,
        TRY_CAST(NULLIF(idade_cliente, '') AS INTEGER) AS idade_cliente,
        NULLIF(genero_cliente, '') AS genero_cliente,
        produto,
        sabor,
        CAST(unidades_vendidas AS INTEGER) AS unidades_vendidas,
        CAST(preco_unitario AS DOUBLE) AS preco_unitario,
        CAST(receita_brl AS DOUBLE) AS receita_brl,
        forma_pagamento,
        CAST(unidades_vendidas AS INTEGER) > 50 AS unidades_suspeitas
    FROM stg_vendas
""")

con.execute("SELECT * FROM stg_vendas WHERE unidades_suspeitas").df()

O outlier de 500 unidades (erro de digitação proposital no PDV) não foi apagado: ficou sinalizado em `unidades_suspeitas`, pra quem for analisar decidir o que fazer com ele.

### Pausa pra praticar

Quantas linhas de `stg_vendas` têm `genero_cliente` nulo? Escreva a consulta SQL (dica: `WHERE genero_cliente IS NULL`).

In [ ]:
# seu código aqui

## 10. Staging em SQL puro: clima

O clima é mais simples de limpar: só precisa tipar os números e a data.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE stg_clima AS
    SELECT
        cidade,
        CAST(data AS DATE) AS data,
        CAST(temp_max_c AS DOUBLE) AS temp_max_c,
        CAST(temp_min_c AS DOUBLE) AS temp_min_c,
        CAST(precipitacao_mm AS DOUBLE) AS precipitacao_mm
    FROM raw.clima
""")

con.execute("SELECT * FROM stg_clima LIMIT 5").df()

## 11. Mart: a pergunta de negócio

Com `stg_vendas` e `stg_clima` prontos, a pergunta central do case (vendemos mais quando está mais quente?) é só um `JOIN` mais uma agregação. `stg_vendas` está no grão de transação; `stg_clima` está no grão de dia/cidade: agregamos as vendas pro grão dia/loja antes de juntar.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE mart_vendas_clima AS
    SELECT
        v.data_venda,
        v.loja_id,
        v.cidade,
        COUNT(*) AS qtd_transacoes,
        SUM(v.unidades_vendidas) AS unidades_vendidas,
        SUM(v.receita_brl) AS receita_total_brl,
        c.temp_max_c,
        c.temp_min_c,
        c.precipitacao_mm,
        CASE
            WHEN c.temp_max_c < 20 THEN '< 20C'
            WHEN c.temp_max_c < 28 THEN '20-28C'
            ELSE '> 28C'
        END AS faixa_temperatura
    FROM stg_vendas v
    LEFT JOIN stg_clima c
        ON v.cidade = c.cidade AND v.data_venda = c.data
    GROUP BY 1, 2, 3, c.temp_max_c, c.temp_min_c, c.precipitacao_mm
""")

mart = con.execute("SELECT * FROM mart_vendas_clima").df()
mart.head()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(mart["temp_max_c"], mart["receita_total_brl"], alpha=0.4, s=18)
ax.set_xlabel("Temperatura máxima do dia (°C)")
ax.set_ylabel("Receita do dia na loja (R$)")
ax.set_title("A pergunta do case, respondida: receita vs. temperatura")
plt.show()

### Pausa pra praticar

Qual cidade teve a maior receita total nos dias de faixa `'> 28C'`? Escreva a consulta em `mart_vendas_clima` (dica: `WHERE faixa_temperatura = '> 28C'`, agrupando por `cidade`).

In [ ]:
# seu código aqui

## 12. Stretch: os outros marts

`scripts/transform.sql` também cria `mart_sabor_temperatura` (sabor mais vendido por faixa de temperatura) e `mart_clientes` (ticket médio e loja favorita por cliente), com a mesma receita de agregação.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE mart_sabor_temperatura AS
    SELECT
        v.produto,
        v.sabor,
        CASE
            WHEN c.temp_max_c < 20 THEN '< 20C'
            WHEN c.temp_max_c < 28 THEN '20-28C'
            ELSE '> 28C'
        END AS faixa_temperatura,
        SUM(v.unidades_vendidas) AS unidades_vendidas,
        SUM(v.receita_brl) AS receita_total_brl
    FROM stg_vendas v
    LEFT JOIN stg_clima c
        ON v.cidade = c.cidade AND v.data_venda = c.data
    GROUP BY 1, 2, 3
""")

con.execute("SELECT * FROM mart_sabor_temperatura ORDER BY unidades_vendidas DESC LIMIT 5").df()

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE mart_clientes AS
    WITH compras_por_loja AS (
        SELECT cliente_id, loja_id, COUNT(*) AS qtd
        FROM stg_vendas
        GROUP BY cliente_id, loja_id
    ),
    loja_favorita AS (
        SELECT cliente_id, loja_id AS loja_favorita
        FROM compras_por_loja
        QUALIFY ROW_NUMBER() OVER (PARTITION BY cliente_id ORDER BY qtd DESC) = 1
    )
    SELECT
        v.cliente_id,
        ANY_VALUE(v.nome_cliente) AS nome_cliente,
        lf.loja_favorita,
        COUNT(*) AS qtd_compras,
        SUM(v.receita_brl) AS receita_total_brl,
        ROUND(AVG(v.receita_brl), 2) AS ticket_medio_brl
    FROM stg_vendas v
    JOIN loja_favorita lf ON lf.cliente_id = v.cliente_id
    GROUP BY v.cliente_id, lf.loja_favorita
""")

con.execute("SELECT * FROM mart_clientes ORDER BY receita_total_brl DESC LIMIT 5").df()

## 13. Resumo final

Fechando com o mesmo resumo que `run_pipeline.py` imprime no final: contagem de linhas de cada tabela do banco.

In [ ]:
tabelas = con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").fetchall()

for schema, tabela in tabelas:
    nome_completo = f"{schema}.{tabela}" if schema != "main" else tabela
    count = con.execute(f"SELECT COUNT(*) FROM {nome_completo}").fetchone()[0]
    print(f"{nome_completo:30s} {count:>6d} linhas")

## 14. Agora é sua vez: perguntas de negócio

Os marts estão prontos. Use `mart_vendas_clima`, `mart_sabor_temperatura` e `mart_clientes` (em SQL ou pandas, como preferir) para responder:

**1.** A loja com a maior receita total (`mart_vendas_clima`) é a mesma loja com o maior ticket médio por transação (`receita_total_brl / qtd_transacoes`)? Calcule os dois rankings e compare.

In [ ]:
# seu código aqui

**2.** Considerando todas as faixas de temperatura juntas, qual é o sabor mais vendido em unidades (`mart_sabor_temperatura`)?

In [ ]:
# seu código aqui

**3.** Quantos clientes (`mart_clientes`) têm `ticket_medio_brl` acima de R$ 50? Qual `loja_favorita` é mais comum entre eles?

In [ ]:
# seu código aqui

**4.** Combinando o que você viu em `mart_vendas_clima` e `mart_sabor_temperatura`: dias mais quentes vendem proporcionalmente mais Picolé do que Sorvete Pote? Escreva uma consulta que sustente sua resposta.

In [ ]:
# seu código aqui

## Para ir além

- Compare o que você fez aqui com os scripts de verdade: `scripts/extract_csv.py`, `scripts/extract_weather.py` e `scripts/transform.sql`. A lógica é a mesma; os scripts têm mais tratamento de erro (cache offline de clima, fallback de coordenadas via geocoding) e são a versão que o `python scripts/run_pipeline.py` roda de verdade.
- **Exercício:** troque os limites da faixa de temperatura (`< 20C`, `20-28C`, `> 28C`) por faixas diferentes (ex.: 4 faixas em vez de 3) e veja como o gráfico da seção 11 muda.
- **Exercício:** `mart_sabor_temperatura` responde "qual sabor vende mais por temperatura". Escreva uma consulta parecida que responda "qual sabor vende mais por loja" (sem considerar temperatura).
- Quando estiver confiante, rode o pipeline de verdade: `python scripts/run_pipeline.py`. Ele usa o banco oficial `workshop.duckdb` (não o banco de sandbox que criamos aqui).

In [ ]:
con.close()